# MDTF ESNB Notebook: North Atlantic Ocean POD

Scaffold for running the `natl_ocean` POD (E. Maroon et al.) through the ESNB interface.

**Before running:**
1. The `diagnostics/natl_ocean/` directory must exist in this repo. If missing, cherry-pick from `emaroon/cmip_input_noamoc`.
2. The catalog `diagnostics/natl_ocean/CMIP_CESM_historical_001.json` + its `.csv` must be present and point at readable glade paths.
3. Run this notebook with the `esnb` kernel (see esnbtest_LH.ipynb cell 0 for kernel setup).

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json
from pathlib import Path
import xarray as xr

os.environ['CODE_ROOT'] = '/glade/work/taydral/MDTF/MDTF-diagnostics'
os.environ['WORK_DIR']  = '/glade/work/taydral/MDTF/wkdir/MDTF_output/natl_ocean'

In [ ]:
module_path = os.getenv('CODE_ROOT')
if module_path not in sys.path:
    sys.path.append(module_path)

from src import util
from src.util import json_utils

## Section 1: POD Settings

Load directly from the POD's `settings.jsonc`. Switch `mode` to `"interactive"` if you want to override any value inline.

In [ ]:
mode = "prod"  # "prod" = read settings.jsonc, "interactive" = use inline dict below
verbose = True

if mode == "prod":
    settings_file_path = os.path.join(module_path, "diagnostics/natl_ocean/settings.jsonc")
    settings_dict = json_utils.read_json(settings_file_path)
else:
    # Interactive override — keep in sync with diagnostics/natl_ocean/settings.jsonc
    settings_dict = {
        "settings": {
            "description": "North Atlantic Ocean diagnostics",
            "driver": "natl_driver.py",
            "long_name": "North Atlantic diagnostic in Subtropical to Subpolar latitudes",
            "convention": "CMIP",
            "runtime_requirements": {"python3": ["matplotlib", "xarray", "xesmf"]},
        },
        "dimensions": {
            "nlat": {"standard_name": "latitude",  "units": "degrees_north", "axis": "Y"},
            "nlon": {"standard_name": "longitude", "units": "degrees_east", "axis": "X"},
            "lev":  {"standard_name": "depth", "units": "centimeters", "axis": "Z", "positive": "down"},
            "time": {"standard_name": "time"},
        },
        "varlist": {
            "tos":       {"frequency": "month", "realm": "ocean", "dimensions": ["time","nlat","nlon"],       "standard_name": "sea_surface_temperature",            "units": "degC"},
            "vsf":       {"frequency": "month", "realm": "ocean", "dimensions": ["time","nlat","nlon"],       "standard_name": "virtual_salt_flux_into_sea_water",    "units": "kg m-2 s-1"},
            "hfds":      {"frequency": "month", "realm": "ocean", "dimensions": ["time","nlat","nlon"],       "standard_name": "surface_downward_heat_flux_in_sea_water", "units": "W m-2"},
            "so":        {"frequency": "month", "realm": "ocean", "dimensions": ["time","lev","nlat","nlon"],"standard_name": "sea_water_salinity",                 "units": "psu"},
            "thetao":    {"frequency": "month", "realm": "ocean", "dimensions": ["time","lev","nlat","nlon"],"standard_name": "sea_water_potential_temperature",    "units": "degC"},
            "volcello":  {"realm": "ocean", "dimensions": ["lev","nlat","nlon"], "standard_name": "ocean_volume", "units": "m3"},
            "areacello": {"realm": "ocean", "dimensions": ["nlat","nlon"],      "standard_name": "cell_area",    "units": "m2"},
        },
    }

print(json.dumps(settings_dict.get("settings", {}), indent=2))

## Section 2: Case / Runtime Config

Defines which simulation(s) to pull from the catalog, the date range, and output paths.

In [ ]:
case_info = {
    "case_list": {
        "CESM2_historical_r1i1p1f1": {
            "model": "CESM",
            "convention": "CMIP",
            "startdate": "1980-01-01",
            "enddate":   "1981-12-31",
        }
    },
    # Intake-esm catalog header (must exist on disk)
    "DATA_CATALOG":     str(Path(module_path) / "diagnostics/natl_ocean/CMIP_CESM_historical_001.json"),
    "OBS_DATA_ROOT":    "/glade/work/taydral/MDTF/inputdata/obs_data",
    "WORK_DIR":         os.environ['WORK_DIR'],
    "OUTPUT_DIR":       os.environ['WORK_DIR'],
    "conda_root":       "/glade/u/home/taydral/miniconda3",
    "conda_env_root":   "/glade/u/home/taydral/miniconda3/envs",
    "micromamba_exe":   "",
    "large_file":       False,
    "make_multicase_figure_html": False,
    "make_variab_tar":  False,
    "overwrite":        True,
    "pod_list":         ["natl_ocean"],
    "run_pp":           True,
    "save_pp_data":     True,
    "save_ps":          False,
    "translate_data":   True,
    "user_pp_scripts":  [""],
}

Path(case_info["WORK_DIR"]).mkdir(parents=True, exist_ok=True)
print(f"Catalog: {case_info['DATA_CATALOG']}")
print(f"Exists:  {Path(case_info['DATA_CATALOG']).exists()}")

## Section 3: Resolve & Load Data via ESNB

In [ ]:
import esnb
from esnb import NotebookDiagnostic, CaseGroup2

pod_env_vars = NotebookDiagnostic(settings_dict)

groups = [
    CaseGroup2(
        case_info,
        date_range=(case_info["case_list"]["CESM2_historical_r1i1p1f1"]["startdate"],
                    case_info["case_list"]["CESM2_historical_r1i1p1f1"]["enddate"]),
    )
]

pod_env_vars.resolve(groups)
pod_env_vars.files

In [ ]:
pod_env_vars.open()
print(f"Datasets loaded: {len(pod_env_vars.datasets)}")
ds_merged = xr.merge(pod_env_vars.datasets, join='outer')
ds_merged

## Section 4: Run POD Diagnostics

Pulls helper functions from `diagnostics/natl_ocean/POD_utils.py` (Maroon et al.). The `natl_driver.py` script
strings these together for batch mode; below we call them step-by-step in the notebook.

In [ ]:
pod_dir = Path(module_path) / 'diagnostics/natl_ocean'
if str(pod_dir) not in sys.path:
    sys.path.append(str(pod_dir))

import POD_utils  # compute_sigma0, compute_mld, compute_zavg, regrid, plot_preproc, error_stats, ...

### 4.1 Derived variables (σ₀, MLD, upper-200m thickness-weighted means)

In [ ]:
# TODO: wire up once POD_utils function signatures are confirmed.
# sigma0 = POD_utils.compute_sigma0(ds_merged['thetao'], ds_merged['so'])
# mld    = POD_utils.compute_mld(sigma0)
# sst_zavg = POD_utils.compute_zavg(ds_merged, 'thetao')

### 4.2 Regrid to 1×1° lat-lon

In [ ]:
# ds_regridded = POD_utils.regrid(ds_merged, method='bilinear')

### 4.3 Climatology, bias vs. obs, error stats, plots

In [ ]:
# clim = ds_regridded.groupby('time.month').mean('time')
# stats = POD_utils.error_stats(clim, obs_ds)
# POD_utils.plot_preproc(...)